In [1]:
from pathlib import Path
from evaluate_baselines import evaluate


In [2]:
checkpoint = Path("baseline_1.pt")
histories = [1, 2, 3, 5, 8, None]  # None means growing history.
models = ["persistence", "wsts_proxy", "ours"]
output_dir = Path("data/evaluation/wsts_overlap")
# WSTS proxy: predicted active fire + deterministic burned-area accumulation.
# Only the three frozen 2020 validation fires and pixels covered by original WSTS inputs.
# One-time input download (already completed): python wsts_baseline.py --download


In [3]:
report = evaluate(histories=histories, checkpoint=checkpoint,
                  models=models, output_dir=output_dir)


persistence 24462753 1 done 0 s
persistence 24462753 2 done 0 s
persistence 24462753 3 done 0 s
persistence 24462753 5 done 0 s
persistence 24462753 8 done 1 s
persistence 24462753 growing done 1 s
persistence 24332622 1 done 1 s
persistence 24332622 2 done 1 s
persistence 24332622 3 done 1 s
persistence 24332622 5 done 1 s
persistence 24332622 8 done 1 s
persistence 24332622 growing done 1 s
persistence 24103571 1 done 1 s
persistence 24103571 2 done 1 s
persistence 24103571 3 done 2 s
persistence 24103571 5 done 2 s
persistence 24103571 8 done 2 s
persistence 24103571 growing done 2 s
wsts_proxy 24462753 1 done 5 s
wsts_proxy 24462753 2 done 6 s
wsts_proxy 24462753 3 done 7 s
wsts_proxy 24462753 5 done 8 s
wsts_proxy 24462753 8 done 9 s
wsts_proxy 24462753 growing done 10 s
wsts_proxy 24332622 1 done 12 s
wsts_proxy 24332622 2 done 12 s
wsts_proxy 24332622 3 done 13 s
wsts_proxy 24332622 5 done 14 s
wsts_proxy 24332622 8 done 15 s
wsts_proxy 24332622 growing done 16 s
wsts_proxy 2410

In [4]:
print("43 next-day forecasts; shared covered pixels, original TS-SatFire resolution.")
print(f"{'History':<10} {'Model':<14} {'Active AP':>10} {'New AP':>10} {'Burned IoU':>12}")
for length in histories:
    history = "growing" if length is None else str(length)
    for model in models:
        scores = {row["output"]: row for row in report["summary"]
                  if row["history"] == history and row["model"] == model}
        print(f"{history:<10} {model:<14} {scores['active']['macro_ap']:>10.3f} "
              f"{scores['new']['macro_ap']:>10.3f} {scores['burned']['pooled_iou']:>12.3f}")


43 next-day forecasts; shared covered pixels, original TS-SatFire resolution.
History    Model           Active AP     New AP   Burned IoU
1          persistence         0.142      0.002        0.947
1          wsts_proxy          0.281      0.096        0.947
1          ours                0.333      0.298        0.948
2          persistence         0.142      0.002        0.947
2          wsts_proxy          0.151      0.042        0.947
2          ours                0.343      0.321        0.947
3          persistence         0.142      0.002        0.947
3          wsts_proxy          0.194      0.074        0.947
3          ours                0.346      0.327        0.947
5          persistence         0.142      0.002        0.947
5          wsts_proxy          0.344      0.235        0.947
5          ours                0.342      0.320        0.947
8          persistence         0.142      0.002        0.947
8          wsts_proxy          0.124      0.111        0.947
8      

In [5]:
for fire, coverage in report["coverage"].items():
    print(f"{fire}: {coverage['fraction']:.1%} of map scored, "
          f"{len(coverage['forecast_dates'])} forecasts")
print("WSTS burned outputs are derived proxies; its learned output is active fire.")
print("WSTS uses its original inputs; ours uses TS-SatFire. Targets and scored pixels match.")
print("Results:", output_dir / "summary.csv")


24462753: 26.5% of map scored, 15 forecasts
24332622: 26.3% of map scored, 15 forecasts
24103571: 30.7% of map scored, 13 forecasts
WSTS burned outputs are derived proxies; its learned output is active fire.
WSTS uses its original inputs; ours uses TS-SatFire. Targets and scored pixels match.
Results: data/evaluation/wsts_overlap/summary.csv
